# **Hands-on Lab: Interactive Visual Analytics with Folium**

Launch success may depend on the location and proximities of a launch site. Here we use `folium` to map every SpaceX launch site, color-code every historical launch by outcome, and analyze each site's proximity to the coastline, nearby roads, and cities.

**Tasks:** (1) mark all launch sites, (2) mark success/failure for every launch, (3) analyze proximities.

In [1]:
import folium
import pandas as pd
from folium.plugins import MarkerCluster, MousePosition
from folium.features import DivIcon
from math import sin, cos, sqrt, atan2, radians

/private/tmp/claude-501/-Users-biancachacon-Documents-PYTHON/3414aaf3-efa4-495a-912b-ce28258331ec/scratchpad/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Task 1: Mark all launch sites on a map

In [2]:
spacex_df = pd.read_csv('spacex_launch_geo.csv')
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


In [3]:
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

for _, site in launch_sites_df.iterrows():
    coordinate = [site['Lat'], site['Long']]
    circle = folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(
        folium.Popup(site['Launch Site']))
    marker = folium.map.Marker(
        coordinate,
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % site['Launch Site'],
        )
    )
    site_map.add_child(circle)
    site_map.add_child(marker)

site_map.save('folium_launch_sites_map.html')
site_map

All four launch sites sit at low latitudes near the US coastline — close to the equator (to gain rotational speed for the launch) and directly on the coast (so spent boosters and any failed launches fall over water, not populated land).

## Task 2: Mark the success/failed launches for each site

In [4]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


In [5]:
marker_cluster = MarkerCluster()

def assign_marker_color(launch_outcome):
    return 'green' if launch_outcome == 1 else 'red'

spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.head()

,Launch Site,Lat,Long,class,marker_color
0,CCAFS LC-40,28.562302,-80.577356,0,red
1,CCAFS LC-40,28.562302,-80.577356,0,red
2,CCAFS LC-40,28.562302,-80.577356,0,red
3,CCAFS LC-40,28.562302,-80.577356,0,red
4,CCAFS LC-40,28.562302,-80.577356,0,red


In [6]:
site_map.add_child(marker_cluster)

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color']),
        popup=f"{record['Launch Site']} — {'Success' if record['class'] == 1 else 'Failure'}"
    )
    marker_cluster.add_child(marker)

site_map.save('folium_launch_sites_map.html')
site_map

Zooming into the marker clusters, **CCAFS SLC-40** and **KSC LC-39A** show a majority of green (successful) markers in their most recent launches, while **CCAFS LC-40**, used for the earliest Falcon 9 flights, shows more red markers — reflecting SpaceX's improving landing success rate over time.

## Task 3: Calculate the distances between a launch site and its proximities

In [7]:
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)
site_map.add_child(mouse_position)

In [8]:
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat / 2) ** 2 + cos(lat1) * cos(lat2) * sin(dlon / 2) ** 2
    c_ = 2 * atan2(sqrt(a), sqrt(1 - a))
    return R * c_

We analyze proximities for **CCAFS SLC-40** (28.563197, -80.576820), reading coastline/road/city coordinates off the interactive map with the mouse-position readout above.

In [9]:
launch_site_lat, launch_site_lon = 28.563197, -80.576820

proximities = {
    'Closest coastline': (28.56367, -80.57163),
    'Samuel C. Phillips Parkway (highway)': (28.56335, -80.57085),
    'Cape Canaveral, FL (nearest city)': (28.40583, -80.60483),
}

for label, (lat, lon) in proximities.items():
    distance = calculate_distance(launch_site_lat, launch_site_lon, lat, lon)
    print(f"{label}: {distance:.2f} km")

    distance_marker = folium.Marker(
        [lat, lon],
        icon=DivIcon(
            icon_size=(20, 20),
            icon_anchor=(0, 0),
            html='<div style="font-size: 12; color:#d35400;"><b>{:.2f} KM</b></div>'.format(distance),
        )
    )
    site_map.add_child(distance_marker)

    lines = folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], [lat, lon]], weight=1, color='#d35400')
    site_map.add_child(lines)

site_map.save('folium_launch_sites_map.html')
site_map

Closest coastline: 0.51 km
Samuel C. Phillips Parkway (highway): 0.58 km
Cape Canaveral, FL (nearest city): 17.72 km


**Findings:** launch sites sit under 1 km from the coastline (so descending stages and any failures fall over the ocean), close to a dedicated access highway, and several kilometers from the nearest town — SpaceX keeps its pads near infrastructure and transport links while maintaining a safe buffer from populated areas.